# Importing Libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import SOAP
from SOAP import PHOENIX
import pandas as pd

Warming up JIT-compiled functions...


# Set Stellar Parameters

In [2]:
# Star-like star, rigid rotation
radius=1                # Stellar radius [solar radii]
mass=1                  # Stellar mass [solar masses]
prot=24.47              # Stellar rotation period [days]
incl=90                 # Stellar inclination [degrees] -> 90 is equator-on, 0 is pole-on
logg= 4.4               # Stellar surface gravity [cgs]
feh= 0.00               # Stellar metallicity [dex]
Teff=5777               # Stellar effective temperature [K]
ldcn=[0.5, 0.16]        # Linear and quadratic limb-darkening coefficients (must agree with λ, see below)
law=1                   # Limb-darkening law (quadratic)
start_psi=0.0           # Starting phase of the star

# Set Observational Parameters

## CHEOPS

In [3]:
# Observational parameters 
λ_CHEOPS=[3300, 11000]    # Wavelength range [Angstrom] 

# load in bandpass
bp_CHEOPS = np.array(pd.read_csv('CHEOPS_bandpass.csv')['PASSBAND'])

## TESS

In [4]:
# Observational parameters 
λ_TESS=[6000, 10000]    # Wavelength range [Angstrom] 

# load in bandpass
bp_TESS = np.array(pd.read_csv('tess-response-function-v2.0.csv', skiprows=6)[' lambda Transmission'])

# Set QS and AR Spectra

In [5]:
spQ_CHEOPS = PHOENIX(wave_range=λ_CHEOPS, normalize=True, teff=Teff, logg=logg, Z=feh) # stellar spectrum
spQ_ar_CHEOPS = PHOENIX(wave_range=λ_CHEOPS, normalize=True, teff=Teff, logg=logg, Z=feh, contrast=-600, ar=True) # AR spectrum
spQ_TESS = PHOENIX(wave_range=λ_TESS, normalize=True, teff=Teff, logg=logg, Z=feh) # stellar spectrum
spQ_ar_TESS = PHOENIX(wave_range=λ_TESS, normalize=True, teff=Teff, logg=logg, Z=feh, contrast=-600, ar=True) # AR spectrum

# Simulate Stellar Spectra

## Create Function

In [6]:
def run_sim_spectra(λ, spQ, spQ_ar): 

    # create simulation variable with 600 grid points 
    # wavelength for temperature contrast via Planck's law is median of above list
    sim = SOAP.Simulation(pixel=spQ, pixel_ar=spQ_ar, wlll=(λ[1]+λ[0])/2, grid=600, active_regions=[])

    # add active region at equator
    sim.active_regions=[SOAP.ActiveRegion(check=True, lon=180, lat=0, size=0.084, temp_diff=600, active_region_type=0)]

    # Set the properties of the star
    sim.star.set(prot=prot,incl=incl,coeffs=ldcn,law=law,start_psi=start_psi,radius=radius,mass=mass,teff=Teff)

    # remove planet
    sim.planet.Rp=0

    # Simulate over 3 stellar phases
    psi = np.linspace(0,3,5)

    # Calculate spectra
    out=sim.calculate_signal(psi, renormalize_rv=False)
    
    return sim, out

## CHEOPS

In [ ]:
# run simulation 
sim_spec_CHEOPS, out_spec_CHEOPS = run_sim_spectra(λ_CHEOPS, spQ_CHEOPS, spQ_ar_CHEOPS)

## TESS